In [4]:
from sdk_client import Robot

rob = Robot("eth0")

In [4]:
import time

import ipywidgets as widgets
from IPython.display import display

from sdk_client import BODY_JOINT_NAME_BY_INDEX, UPPER_BODY_JOINTS, WAIST_HOLD_KD, WAIST_HOLD_KP

JOINT_LIMITS = {
    12: (-2.618, 2.618),   # waist.yaw
    13: (-0.52, 0.52),     # waist.roll
    14: (-0.52, 0.52),     # waist.pitch
    15: (-3.0892, 2.6704), # left_arm.shoulder_pitch
    16: (-1.5882, 2.2515), # left_arm.shoulder_roll
    17: (-2.618, 2.618),   # left_arm.shoulder_yaw
    18: (-1.0472, 2.0944), # left_arm.elbow
    19: (-1.9722, 1.9722), # left_arm.wrist_roll
    20: (-1.6144, 1.6144), # left_arm.wrist_pitch
    21: (-1.6144, 1.6144), # left_arm.wrist_yaw
    22: (-3.0892, 2.6704), # right_arm.shoulder_pitch
    23: (-2.2515, 1.5882), # right_arm.shoulder_roll
    24: (-2.618, 2.618),   # right_arm.shoulder_yaw
    25: (-1.0472, 2.0944), # right_arm.elbow
    26: (-1.9722, 1.9722), # right_arm.wrist_roll
    27: (-1.6144, 1.6144), # right_arm.wrist_pitch
    28: (-1.6144, 1.6144), # right_arm.wrist_yaw
}

RATE_HZ = 50.0
MAX_SPEED_RAD_S = 0.45
KP = 30.0
KD = 1.5
WAIST_KP = WAIST_HOLD_KP
WAIST_KD = WAIST_HOLD_KD
TIMEOUT_S = 3.0

joint_options = [
    (f"{BODY_JOINT_NAME_BY_INDEX[joint]} ({joint})", joint)
    for joint in UPPER_BODY_JOINTS
]

joint_dropdown = widgets.Dropdown(
    options=joint_options,
    value=22,
    description="Joint",
    layout=widgets.Layout(width="420px"),
)
position_slider = widgets.FloatSlider(
    description="Position",
    min=JOINT_LIMITS[22][0],
    max=JOINT_LIMITS[22][1],
    step=0.01,
    value=0.0,
    continuous_update=False,
    readout_format=".3f",
    layout=widgets.Layout(width="620px"),
)
refresh_button = widgets.Button(description="Read current")
status = widgets.Output()
_updating_slider = False


def _read_upper_body_positions():
    return rob._read_joint_positions_or_raise(UPPER_BODY_JOINTS, timeout=TIMEOUT_S)


def _set_slider_to_joint(joint_index):
    global _updating_slider
    positions = _read_upper_body_positions()
    lo, hi = JOINT_LIMITS[int(joint_index)]
    value = max(lo, min(hi, float(positions[int(joint_index)])))
    _updating_slider = True
    try:
        position_slider.min = lo
        position_slider.max = hi
        position_slider.value = value
    finally:
        _updating_slider = False
    with status:
        status.clear_output()
        print(f"{BODY_JOINT_NAME_BY_INDEX[int(joint_index)]}: current {value:.3f} rad")


def _ramp_joint(joint_index, target):
    positions = _read_upper_body_positions()
    joint_index = int(joint_index)
    start = float(positions[joint_index])
    target = float(target)
    with status:
        status.clear_output()
        print(f"Ramping {BODY_JOINT_NAME_BY_INDEX[joint_index]}: {start:.3f} -> {target:.3f} rad")
    rob.move_upper_body_joint(
        joint_index,
        target,
        command_rate_hz=RATE_HZ,
        max_speed_rad_s=MAX_SPEED_RAD_S,
        kp=KP,
        kd=KD,
        waist_kp=WAIST_KP,
        waist_kd=WAIST_KD,
        timeout=TIMEOUT_S,
    )
    with status:
        status.clear_output()
        print(f"Done: {BODY_JOINT_NAME_BY_INDEX[joint_index]} = {target:.3f} rad")


def _on_joint_change(change):
    if change["name"] == "value":
        _set_slider_to_joint(change["new"])


def _on_slider_change(change):
    if _updating_slider or change["name"] != "value":
        return
    _ramp_joint(joint_dropdown.value, change["new"])


def _on_refresh(_button):
    _set_slider_to_joint(joint_dropdown.value)


joint_dropdown.observe(_on_joint_change, names="value")
position_slider.observe(_on_slider_change, names="value")
refresh_button.on_click(_on_refresh)

_set_slider_to_joint(joint_dropdown.value)
display(widgets.VBox([joint_dropdown, position_slider, refresh_button, status]))

In [5]:
import ipywidgets as widgets
from IPython.display import display

from sdk_hand import hand_mid_targets, hand_open_targets

HAND_RATE_HZ = 50.0
HAND_HOLD_S = 0.7
HAND_RAMP_S = 0.6
HAND_KP = 0.8
HAND_KD = 0.05
HAND_TAU = 0.0

hand_dropdown = widgets.Dropdown(
    options=[("Left hand", "left"), ("Right hand", "right")],
    value="right",
    description="Hand",
    layout=widgets.Layout(width="260px"),
)
grip_slider = widgets.FloatSlider(
    description="Grip",
    min=0.0,
    max=1.0,
    step=0.05,
    value=0.0,
    continuous_update=False,
    readout=False,
    layout=widgets.Layout(width="520px"),
)
grip_label = widgets.Label("Closed")
hand_status = widgets.Output()
_updating_grip = False


def _blend_hand_targets(hand, openness):
    closed = hand_mid_targets(hand)
    opened = hand_open_targets(hand)
    alpha = max(0.0, min(1.0, float(openness)))
    return [c + (o - c) * alpha for c, o in zip(closed, opened)]


def _set_grip_label(value):
    grip_label.value = "Closed" if value <= 0.0 else ("Open" if value >= 1.0 else f"{value:.0%} open")


def _apply_grip(hand, openness):
    targets = _blend_hand_targets(hand, openness)
    with hand_status:
        hand_status.clear_output()
        print(f"Moving {hand} hand to {_set_grip_text(openness)}")
    controller = rob._get_hand(hand)
    if controller._last_targets is None:
        controller._last_targets = hand_mid_targets(hand)
    controller.set_targets(
        targets,
        hold_s=HAND_HOLD_S,
        rate_hz=HAND_RATE_HZ,
        kp=HAND_KP,
        kd=HAND_KD,
        tau=HAND_TAU,
        ramp_s=HAND_RAMP_S,
    )
    with hand_status:
        hand_status.clear_output()
        print(f"Done: {hand} hand {_set_grip_text(openness)}")


def _set_grip_text(value):
    value = float(value)
    if value <= 0.0:
        return "closed"
    if value >= 1.0:
        return "open"
    return f"{value:.0%} open"


def _on_hand_change(change):
    global _updating_grip
    if change["name"] != "value":
        return
    _updating_grip = True
    try:
        grip_slider.value = 0.0
        _set_grip_label(0.0)
    finally:
        _updating_grip = False
    with hand_status:
        hand_status.clear_output()
        print(f"Selected {change['new']} hand. Slider is at closed.")


def _on_grip_change(change):
    if _updating_grip or change["name"] != "value":
        return
    _set_grip_label(change["new"])
    _apply_grip(hand_dropdown.value, change["new"])


hand_dropdown.observe(_on_hand_change, names="value")
grip_slider.observe(_on_grip_change, names="value")
_set_grip_label(grip_slider.value)
display(widgets.VBox([hand_dropdown, widgets.HBox([grip_slider, grip_label]), hand_status]))

In [3]:
rob.get_joint_positions()

{'left_leg.hip_pitch': -0.35701698064804077,
 'left_leg.hip_roll': -0.0317789688706398,
 'left_leg.hip_yaw': 0.014543598517775536,
 'left_leg.knee': 0.5870459079742432,
 'left_leg.ankle_pitch': -0.29755812883377075,
 'left_leg.ankle_roll': 0.019886313006281853,
 'right_leg.hip_pitch': -0.4050987660884857,
 'right_leg.hip_roll': 0.030364297330379486,
 'right_leg.hip_yaw': -0.022752830758690834,
 'right_leg.knee': 0.6260346174240112,
 'right_leg.ankle_pitch': -0.27478697896003723,
 'right_leg.ankle_roll': -0.029174692928791046,
 'waist.yaw': 0.0017275545978918672,
 'waist.roll': 0.0011074466165155172,
 'waist.pitch': 0.02886468544602394,
 'left_arm.shoulder_pitch': 0.10206964612007141,
 'left_arm.shoulder_roll': 0.325467586517334,
 'left_arm.shoulder_yaw': -0.06630872189998627,
 'left_arm.elbow': 0.29936593770980835,
 'left_arm.wrist_roll': 0.4135396480560303,
 'left_arm.wrist_pitch': -0.6038970947265625,
 'left_arm.wrist_yaw': -0.02731204964220524,
 'right_arm.shoulder_pitch': 0.0141054